# Dataset v3: temporal masking for the pretrained LRW MS-TCN backend

Controlled experiment: keep the dataset, frozen LRW visual features, horizontal
flip, held-out speakers, seeds, optimizer, and metrics fixed. Add temporal masking
only to training sequences before the official LRW-pretrained multi-scale TCN,
then train a new 10-word linear classifier. Validation stays unmasked.

The final test split is never loaded.

## 1. Connect Google Drive and install the small dependency

In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=True, timeout_ms=300000)
%pip -q install gdown

## 2. Restore metadata, feature caches, and the official checkpoint

In [ ]:
from pathlib import Path
import random
import shutil
import subprocess
import sys
import time
import zipfile

import gdown
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from torch import nn
from torch.utils.data import DataLoader, Dataset


def load_torch_file(path, device='cpu'):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


drive_root = Path('/content/drive/MyDrive/lipreading_sem4')
upstream_dir = Path('/content/Lipreading_using_Temporal_Convolutional_Networks')
upstream_store = drive_root / 'upstream_lrw_pretrained'
upstream_store.mkdir(parents=True, exist_ok=True)

if not upstream_dir.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/mpc001/Lipreading_using_Temporal_Convolutional_Networks.git',
        str(upstream_dir),
    ], check=True)

checkpoint_path = upstream_store / 'lrw_resnet18_mstcn_video.pth'
if not checkpoint_path.exists():
    downloaded = gdown.download(
        id='1vqMpxZ5LzJjg50HlZdj_QFJGm2gQmDUD',
        output=str(checkpoint_path),
        quiet=False,
    )
    if downloaded is None:
        raise RuntimeError('Official LRW checkpoint download failed.')

extract_dir = Path('/content/lipreading_data_v3')
relative_split_dir = (
    Path('03_splits')
    / 'ml_splits_mouth_crops_landmark_dataset_v3'
    / 'speaker_top10'
)
split_dir = extract_dir / 'ru_dataset' / relative_split_dir
if not split_dir.exists():
    archives = list(Path('/content/drive').rglob(
        'colab_lipreading_dataset_v3_train_val.zip'
    ))
    if not archives:
        raise FileNotFoundError(
            'Upload colab_lipreading_dataset_v3_train_val.zip to Google Drive.'
        )
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0], 'r') as archive:
        archive.extractall(extract_dir)

sys.path.insert(0, str(upstream_dir))
train_df = pd.read_csv(split_dir / 'train.csv')
vocab = (split_dir / 'vocab.txt').read_text(encoding='utf-8').splitlines()
word_to_idx = {word: index for index, word in enumerate(vocab)}

feature_root = drive_root / 'controlled_lrw_frontend_bigru_dataset_v3'
original_cache_path = feature_root / 'dataset_v3_train_lrw_resnet18_features.pt'
hflip_cache_path = feature_root / 'dataset_v3_train_lrw_resnet18_hflip_features.pt'
if not original_cache_path.exists() or not hflip_cache_path.exists():
    raise FileNotFoundError(
        'Original or hflip LRW feature cache is missing. Run the cross-speaker '
        'hflip notebook first.'
    )

original_cache = load_torch_file(original_cache_path)
hflip_cache = load_torch_file(hflip_cache_path)
assert len(train_df) == 1901
assert list(original_cache['words']) == train_df['word'].tolist()
assert list(hflip_cache['words']) == train_df['word'].tolist()
assert original_cache['features'].shape == hflip_cache['features'].shape

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device != 'cuda':
    raise RuntimeError('Enable a GPU runtime before running this experiment.')
output_dir = drive_root / 'lrw_pretrained_mstcn_transfer_cross_speaker_dataset_v3'
output_dir.mkdir(parents=True, exist_ok=True)

print('device:', device)
print('features:', original_cache['features'].shape)
print('train speakers:', sorted(train_df['speaker_id'].unique()))
print('vocab:', vocab)
print('output:', output_dir)
print('TEST SPLIT IS NOT LOADED.')

## 3. Convert visual sequences into pretrained MS-TCN embeddings and masked variants

In [ ]:
from lipreading.model import Lipreading


tcn_options = {
    'num_layers': 4,
    'kernel_size': [3, 5, 7],
    'dropout': 0.2,
    'dwpw': False,
    'width_mult': 1,
}
pretrained_model = Lipreading(
    modality='video',
    num_classes=500,
    tcn_options=tcn_options,
    backbone_type='resnet',
    relu_type='swish',
    width_mult=1.0,
    use_boundary=False,
    extract_feats=False,
).to(device)
checkpoint = load_torch_file(checkpoint_path)
pretrained_model.load_state_dict(
    checkpoint.get('model_state_dict', checkpoint), strict=True
)
for parameter in pretrained_model.parameters():
    parameter.requires_grad = False
pretrained_model.eval()


def extract_pretrained_tcn_embeddings(features, batch_size=64):
    batches = []
    with torch.inference_mode():
        for start in range(0, len(features), batch_size):
            batch = features[start:start + batch_size].float().to(device)
            # Official MS-TCN expects [batch, channels, frames].
            sequence = pretrained_model.tcn.mb_ms_tcn(batch.transpose(1, 2))
            pooled = sequence.mean(dim=2)
            batches.append(pooled.cpu())
    return torch.cat(batches)


original_tcn_path = feature_root / 'dataset_v3_train_lrw_mstcn_pooled.pt'
hflip_tcn_path = feature_root / 'dataset_v3_train_lrw_mstcn_hflip_pooled.pt'
if original_tcn_path.exists() and hflip_tcn_path.exists():
    original_tcn = load_torch_file(original_tcn_path)['features'].float()
    hflip_tcn = load_torch_file(hflip_tcn_path)['features'].float()
    print('Loaded cached pretrained MS-TCN embeddings.')
else:
    original_tcn = extract_pretrained_tcn_embeddings(original_cache['features'])
    hflip_tcn = extract_pretrained_tcn_embeddings(hflip_cache['features'])
    torch.save({'features': original_tcn}, original_tcn_path)
    torch.save({'features': hflip_tcn}, hflip_tcn_path)
    print('Saved pretrained MS-TCN embeddings.')

def extract_temporally_masked_embeddings(features, seed, batch_size=64, mask_probability=0.5, mask_width=4):
    generator = torch.Generator().manual_seed(seed)
    masked_batches = []
    with torch.inference_mode():
        for start in range(0, len(features), batch_size):
            batch = features[start:start + batch_size].float().clone()
            for row in range(len(batch)):
                if torch.rand((), generator=generator).item() < mask_probability:
                    begin = int(torch.randint(0, batch.shape[1] - mask_width + 1, (1,), generator=generator))
                    batch[row, begin:begin + mask_width] = 0.0
            batch = batch.to(device)
            sequence = pretrained_model.tcn.mb_ms_tcn(batch.transpose(1, 2))
            masked_batches.append(sequence.mean(dim=2).cpu())
    return torch.cat(masked_batches)

original_masked_tcn = extract_temporally_masked_embeddings(original_cache['features'], seed=1729)
hflip_masked_tcn = extract_temporally_masked_embeddings(hflip_cache['features'], seed=1730)

assert original_tcn.shape == hflip_tcn.shape == original_masked_tcn.shape == hflip_masked_tcn.shape
assert original_tcn.shape[0] == len(train_df)
print('pooled embeddings:', original_tcn.shape)
print('temporal masking: width=4 frames, probability=0.5; validation remains unmasked.')


## 4. Cross-speaker linear-probe comparison against unmasked MS-TCN

In [ ]:
FOLD_SPEAKERS = ['spk03', 'spk17', 'spk19']
SEEDS = [42, 123, 2026]
UNMASKED_MSTCN = {
    ('spk03', 42): 0.698465,
    ('spk03', 123): 0.657397,
    ('spk03', 2026): 0.699259,
    ('spk17', 42): 0.547482,
    ('spk17', 123): 0.525563,
    ('spk17', 2026): 0.551206,
    ('spk19', 42): 0.537798,
    ('spk19', 123): 0.504027,
    ('spk19', 2026): 0.527397,
}


def set_reproducible_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


class VectorDataset(Dataset):
    def __init__(self, features, words):
        self.features = features.float()
        self.labels = torch.tensor(
            [word_to_idx[word] for word in words], dtype=torch.long
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]


class LinearProbe(nn.Module):
    def __init__(self, input_dim, num_classes, dropout=0.3):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(input_dim, num_classes),
        )

    def forward(self, features):
        return self.classifier(features)


def evaluate(model, loader, criterion):
    model.eval()
    labels = []
    predictions = []
    loss_sum = 0.0
    with torch.no_grad():
        for features, targets in loader:
            features = features.to(device)
            targets = targets.to(device)
            logits = model(features)
            loss_sum += criterion(logits, targets).item() * targets.size(0)
            labels.extend(targets.cpu().tolist())
            predictions.extend(logits.argmax(dim=1).cpu().tolist())
    return {
        'loss': loss_sum / len(labels),
        'accuracy': accuracy_score(labels, predictions),
        'macro_f1': f1_score(labels, predictions, average='macro', zero_division=0),
        'balanced_accuracy': balanced_accuracy_score(labels, predictions),
        'predicted_classes': len(set(predictions)),
    }


def run_fold(fold_speaker, seed, num_epochs=50, patience=10):
    set_reproducible_seed(seed)
    is_validation = train_df['speaker_id'].eq(fold_speaker).to_numpy()
    train_indices = np.flatnonzero(~is_validation)
    validation_indices = np.flatnonzero(is_validation)
    train_features = torch.cat([
        original_tcn[train_indices], hflip_tcn[train_indices],
        original_masked_tcn[train_indices], hflip_masked_tcn[train_indices]
    ])
    train_words = (
        train_df.iloc[train_indices]['word'].tolist()
        + train_df.iloc[train_indices]['word'].tolist()
        + train_df.iloc[train_indices]['word'].tolist()
        + train_df.iloc[train_indices]['word'].tolist()
    )
    validation_features = original_tcn[validation_indices]
    validation_words = train_df.iloc[validation_indices]['word'].tolist()

    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(
        VectorDataset(train_features, train_words),
        batch_size=64,
        shuffle=True,
        generator=generator,
        num_workers=0,
    )
    validation_loader = DataLoader(
        VectorDataset(validation_features, validation_words),
        batch_size=64,
        shuffle=False,
        num_workers=0,
    )

    model = LinearProbe(original_tcn.shape[1], len(vocab)).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    best_f1 = -1.0
    best_loss = float('inf')
    best_epoch = 0
    best_state = None
    without_improvement = 0
    started = time.perf_counter()

    for epoch in range(1, num_epochs + 1):
        model.train()
        for features, targets in train_loader:
            features = features.to(device)
            targets = targets.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(features), targets)
            loss.backward()
            optimizer.step()

        metrics = evaluate(model, validation_loader, criterion)
        improved = (
            metrics['macro_f1'] > best_f1
            or (metrics['macro_f1'] == best_f1 and metrics['loss'] < best_loss)
        )
        if improved:
            best_f1 = metrics['macro_f1']
            best_loss = metrics['loss']
            best_epoch = epoch
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            without_improvement = 0
        else:
            without_improvement += 1
        if without_improvement >= patience:
            break

    model.load_state_dict(best_state)
    best_metrics = evaluate(model, validation_loader, criterion)
    unmasked_mstcn_f1 = UNMASKED_MSTCN[(fold_speaker, seed)]
    return {
        'fold_speaker': fold_speaker,
        'seed': seed,
        'train_examples': len(train_features),
        'validation_examples': len(validation_features),
        'best_epoch': best_epoch,
        'epochs_completed': epoch,
        'validation_loss': best_metrics['loss'],
        'validation_accuracy': best_metrics['accuracy'],
        'validation_macro_f1': best_metrics['macro_f1'],
        'validation_balanced_accuracy': best_metrics['balanced_accuracy'],
        'predicted_classes': best_metrics['predicted_classes'],
        'unmasked_mstcn_macro_f1': unmasked_mstcn_f1,
        'delta_masked_minus_unmasked': best_metrics['macro_f1'] - unmasked_mstcn_f1,
        'training_seconds': time.perf_counter() - started,
    }, best_state


results = []
states = {}
for fold_speaker in FOLD_SPEAKERS:
    print(f'\n=== held-out speaker: {fold_speaker} ===')
    for seed in SEEDS:
        result, state = run_fold(fold_speaker, seed)
        results.append(result)
        states[f'{fold_speaker}_seed{seed}'] = state
        print(
            f"seed={seed}: masked MS-TCN F1={result['validation_macro_f1']:.4f}, "
            f"unmasked MS-TCN F1={result['unmasked_mstcn_macro_f1']:.4f}, "
            f"delta={result['delta_masked_minus_unmasked']:+.4f}",
            flush=True,
        )

results_df = pd.DataFrame(results)
summary = {
    'folds': len(FOLD_SPEAKERS),
    'seeds_per_fold': len(SEEDS),
    'unmasked_mstcn_macro_f1_mean': float(results_df.unmasked_mstcn_macro_f1.mean()),
    'masked_mstcn_macro_f1_mean': float(results_df.validation_macro_f1.mean()),
    'paired_delta_mean': float(results_df.delta_masked_minus_unmasked.mean()),
    'paired_delta_std': float(results_df.delta_masked_minus_unmasked.std(ddof=1)),
    'masked_mstcn_wins': int((results_df.delta_masked_minus_unmasked > 0).sum()),
    'comparisons': len(results_df),
}

results_df.to_csv(output_dir / 'pretrained_mstcn_temporal_masking_runs.csv', index=False)
pd.DataFrame([summary]).to_csv(
    output_dir / 'pretrained_mstcn_temporal_masking_summary.csv', index=False
)
torch.save(
    {
        'results': results,
        'states': states,
        'vocab': vocab,
        'test_used': False,
    },
    output_dir / 'pretrained_mstcn_temporal_masking_linear_probes.pt',
)

display(results_df)
print('summary:', summary)
keep_masked_mstcn = summary['paired_delta_mean'] > 0 and summary['masked_mstcn_wins'] >= 5
print('DECISION:', 'KEEP MASKED MS-TCN' if keep_masked_mstcn else 'KEEP UNMASKED MS-TCN')
print('TEST SPLIT WAS NOT LOADED OR USED.')

## 5. Decision gate

Keep the pretrained MS-TCN transfer only if its mean paired macro-F1 gain over
the unmasked MS-TCN baseline is positive and it wins at least 5 of 9 fold-seed comparisons.
Otherwise the simpler unmasked MS-TCN remains the candidate model.

Do not inspect the final test set during this decision.